# EMERGING parser walkthrough

EMERGING combines detailed product and service accounts with a broad geographical coverage.
This notebook shows how to read its MATLAB bundles with MARIO, choose a release and year,
and load the matching fuel-specific CO2 extension when it is available.

The standard releases and the electricity-disaggregated EMERGING-E variant share the same
parser. Their native classifications are kept separate, so changing a version also changes
the sector set available for analysis.


## What this notebook covers

- which official Zenodo version records are relevant for MARIO;
- how local file naming works for `v1.0`, `v2.x`, and `variant="E"` bundles;
- how `download_emerging(...)` maps to the supported version records;
- how `year=`, `regions=`, `load_co2=`, `co2_path=`, and `variant=` are used;
- which caveats matter for very large EMERGING and EMERGING-E databases.


## Versions and available local years

The browser uses the source release as **Version**. All current readings are monetary IOTs,
so those fixed properties do not add version labels or unnecessary filters.

| Version | Years in the current local collection | Parser selection |
| --- | --- | --- |
| [2.2](https://doi.org/10.5281/zenodo.19461860) | 2015, 2018, 2021, 2023 | `variant="standard"`, folder `2.2` |
| [2.0](https://doi.org/10.5281/zenodo.17557778) | 2018 | `variant="standard"`, folder `2.0` |
| [1.0](https://doi.org/10.5281/zenodo.10956623) | 2010, 2015–2019 | `variant="standard"`, folder `1.0` |
| [E (EMERGING-E)](https://doi.org/10.5281/zenodo.18303090) | 2018 | `variant="E"` |

The 2.0 local collection is incomplete: companion CO2 files for 2015 and 2021 do not add
economic years without the matching MRIO bundle. These are verified local years, rather
than a claim about every year published by the providers.

For native regions, sectors and other sets, see [Database scope](../../../resources/data-explorer/scope.rst)
and the downloadable Excel inventories linked above.


For MATLAB v7.3 files, MARIO normalizes the stored axes while the economic matrix is sparse. No manual transpose of `Z` is needed.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_emerging(...)`

The parser currently supports only the multiregional `IOT` workflow.


## Key arguments

The key public arguments are:

- `path`: one EMERGING `.mat` file or a directory containing multiple yearly bundles;
- `table`: currently only `"IOT"` is supported;
- `year`: use it when one directory contains more than one EMERGING year;
- `regions`: optional ISO3 subset to keep only one manageable part of the database;
- `load_co2`: enable or disable automatic companion CO2 import;
- `co2_path`: explicit path to one companion CO2 file when auto-detection is not enough;
- `labels_path`: optional Excel workbook with explicit sector labels; for `variant="E"` it overrides workbook auto-detection next to the bundle;
- `variant`: keep `"standard"` for the original EMERGING releases and use `"E"` for the EMERGING-E power-disaggregated bundle.


## Local file naming conventions

In practice MARIO accepts these local naming conventions:

- `global_mrio_<year>.mat` for `v1.0`;
- `EMERGING_V2_<year>_m.mat` for `v2.x`;
- `EMERGING_V2_<year>.mat` for older local `v2.x` copies when the internal MATLAB structure is still compatible;
- `EMERGING_E_<year>.mat` for the EMERGING-E power-disaggregated variant.

For local `v2.x` files, MARIO does not try to infer the exact sub-version `2.0` versus `2.1` versus `2.2` from the filename alone, because the public naming convention is shared across those releases.


## Expected path structure

`path` can point either to one yearly `.mat` file or to a directory containing the yearly EMERGING bundle and, optionally, the companion CO2 file.

Typical directory layout:

```text
EMERGING/2.2/
├── EMERGING_V2_2023_m.mat
└── EMERGING_CO2_2023.mat
```

For the older `v1.0` naming convention, the economic bundle is typically named like `global_mrio_<year>.mat`.
For EMERGING-E, the local bundle is typically named like `EMERGING_E_<year>.mat` and may ship a sibling `Figure data.xlsx` workbook that MARIO uses to recover ISO3 region codes.

When `path` is a directory, use `year=` to select the economic bundle. With `load_co2=True`, MARIO first looks for the matching CO2 file in the same directory, for example `EMERGING_CO2_2023.mat` for `year=2023`. Use `co2_path=` when the companion file is stored elsewhere or has a non-standard name.

## Download workflow

Use `mario.download_emerging(...)` when you want MARIO to fetch one of the supported official version records, including the newer `version="E"` EMERGING-E record.

`latest` currently resolves to `v2.2`.


In [1]:
import mario

## Download one official version explicitly


In [ ]:
mario.download_emerging(
    path="/path/to/2.2",
    version="2.2",
    years=[2023],
)

{'source': 'https://doi.org/10.5281/zenodo.10956622',
 'version': '2.2',
 'version_record': 'https://doi.org/10.5281/zenodo.19461860',
 'download_dir': '/path/to/emerging_directory',
 'years': [2023],
 'files': ['/path/to/emerging_directory/EMERGING_CO2_2023.mat',
  '/path/to/EMERGING_V2_2023_m.mat']}

## Download the EMERGING-E record

Use `version="E"` when you want MARIO to fetch the public EMERGING-E Zenodo record. With `years=[2018]`, MARIO keeps the main `.mat` file and the companion `Figure data.xlsx` workbook used to recover ISO3 region codes.


In [ ]:
mario.download_emerging(
    path="/path/to/emerging-e",
    version="E",
    years=[2018],
)

Also for EMERGING-E, you can use the `co2_path` argument, passing the path of the CO2 emissions files from standard EMERGING releases.

## Parse one local EMERGING bundle


In [3]:
db = mario.parse_emerging(
    path="/path/to/EMERGING_V2_2023_m.mat",
    table="IOT",
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.
INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.
INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.
INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Parse one local EMERGING-E bundle

`EMERGING-E` is currently a local-file workflow. Use `variant="E"` and point `path` to `EMERGING_E_<year>.mat`.

In [ ]:
db_e = mario.parse_emerging(
    path="/path/to/EMERGING_E_2018.mat",
    table="IOT",
    variant="E"
)

db_e

INFO Parser: reading EMERGING-E bundle EMERGING_E_2018.mat.
INFO Parser: EMERGING-E parsed with 245 regions, 146 sectors, 735 final-demand columns and 1 satellite rows.
INFO Metadata: initialized.


name = EMERGING-E 2018
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 1
Consumption category = 3
Region = 245
Sector = 146

## Parse from a directory and select one year


In [3]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Restrict the region set

The full EMERGING matrix is very large, so `regions=` is often the right first step.


In [4]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
    regions=["DEU", "FRA", "ITA"],
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 3 regions, 133 sectors, 9 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 3
Sector = 133

## Control CO2 loading explicitly


In [5]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
    co2_path="/path/to/EMERGING_CO2_2023.mat",
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Reading the economic and emissions accounts

`V` contains the source value-added row, in current million US dollars. Sum its columns
for a region or the whole model, and multiply by `1e6` when expressing the result in US dollars.
The available bundles do not expose a separate, verified product-tax bridge, so the browser
also presents the native `va` sum as a **GDP proxy**, without an invented tax adjustment.
A note beneath the chart explains that equivalence to GDP at market prices is unverified.

The standard companion files contain seven fossil CO2 accounts, one per fuel. MARIO
imports them into `E`, preserving the native unit and order. They are not a complete GHG
inventory and do not supply direct final-demand emissions. The browser presents their sum
as a **CO₂-only proxy under GHG**, with a visible caveat: CH₄, N₂O and F-gases are absent.
The native Mt CO₂eq values are multiplied by `1e6` to obtain tonnes; CO₂ has GWP 1
under both AR5 and AR6. EMERGING-E in this collection has economic accounts only.

- EMERGING currently supports `IOT` tables.
- The containing release directory identifies 2.0 versus 2.2; their economic filenames are shared.
- `regions=` can restrict the parse when the full model is unnecessary.
- Use a same-year, same-release CO2 file. `load_co2=False` reads only the economic accounts.
- ANT and SER retain historical grouping labels; TMP is the historical code mapped to TLS.
  Current memberships of the historical groups are not inferred.


In [6]:
db.table_type

'IOT'

In [7]:
db.get_index("Region")[:10]

['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ANT', 'BES', 'CUW', 'ARE']

In [8]:
db.get_index("Satellite account")

['Coal',
 'Natural gas',
 'Oil products',
 'Crude, NGL, Ref Feeds.',
 'Other',
 'Oil shale & oil sands',
 'Peat & Peat products']

In [9]:
db.meta.source

'EMERGING MATLAB bundle via Zenodo; (concept DOI https://doi.org/10.5281/zenodo.10956622); Huo, J., Chen, P., Hubacek, K., Zheng, H., Meng, J., & Guan, D. (2022). Full-scale, near real-time multi-regional input-output table for the global emerging economies (EMERGING). Journal of Industrial Ecology, 26, 1218–1232. https://doi.org/10.1111/jiec.13264'